# Veri Hazırlama

In [11]:
import sys
import os


sys.path.insert(0, os.path.abspath(os.getcwd()))

from olist.data import Olist
data = Olist().get_data()
data['sellers'].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [1]:
# "magic commands" to enable autoreload of your imported packages
%load_ext autoreload
%autoreload 2

Amacımız, 9 `.csv` dosyasının tamamını tek bir sözlüğe (dict) `data` adında 9 `pandas.DataFrame` olarak yüklemektir. Burada:
- her **anahtar**, csv dosyasının **temizlenmiş adıdır**
- her **değer**, csv dosyasından oluşturulan **DataFrame**tir

```python
data = { 
    'sellers': DataFrame1,
    'orders': DataFrame2,
    ...
    }
```

### 1. `csv_path` değişkenini oluşturun, bu değişken `"csv" klasörü`nün yolunu bir string olarak depolayacaktır

Yolları kolayca yönetmek için Python'ın `pathlib` modülünü kullanalım. Pathlib çok kullanışlı bir `Path` sınıfı tanımlar.

Bir `Path` örneğini yolumuzu tanımlayan bir string vererek oluşturabiliriz.

In [2]:
from pathlib import Path
csv_path = Path("~/.workintech/olist/data/csv").expanduser()
csv_path

PosixPath('/home/bariscan/.workintech/olist/data/csv')

_`PosixPath`_? Bu, yolun Posix formatında olduğu anlamına gelir. POSIX (Portable Operating System Interface), Unix benzeri işletim sistemleri için uyumluluk ve birlikte çalışabilirliği sağlamak amacıyla standartlaştırılmış API'ler ve kuralların bir setidir. Hem Linux (örneğin WSL'deki Ubuntu) hem de macOS, POSIX uyumludur.

`expanduser()` kullanarak `~` işaretini ana klasörümüzün gerçek mutlak yoluna genişletiyoruz.

Şimdi klasörümüzdeki dosyaları listelemek için `iterdir()` metodunu kullanabiliriz:

In [3]:
file_paths = list(csv_path.iterdir())
file_paths

[PosixPath('/home/bariscan/.workintech/olist/data/csv/product_category_name_translation.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_products_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_customers_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_order_payments_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_geolocation_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_order_items_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_orders_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_order_reviews_dataset.csv'),
 PosixPath('/home/bariscan/.workintech/olist/data/csv/olist_sellers_dataset.csv')]

In [4]:
# Kodunuzu aşağıda test edin. Dizindeki ilk csv dosyasını yüklemeyi deneyin
import pandas as pd
pd.read_csv(file_paths[0]).head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### 2. csv dizinindeki tüm csv dosya adlarını içeren `file_names` listesini oluşturun

- Bunun böyle görünmesi gerekir: `file_names = ['olist_sellers_dataset.csv', ....]`
- `file_paths` adresinden başlayabilirsiniz
- Bir `Path` nesnesi `name` adında bir özelliğe sahiptir

In [5]:
file_names = [path.name for path in file_paths]
file_names

['product_category_name_translation.csv',
 'olist_products_dataset.csv',
 'olist_customers_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_sellers_dataset.csv']

### 3. Sözlük anahtarları listesi `key_names` oluşturun
file_names adresinden başlayarak:
- ".csv" sonekini (varsa) kaldırın
- "_dataset.csv" sonekini (varsa) kaldırın
- "olist_" önekini (varsa) kaldırın

<details>
    <summary>- İpucu - </summary>

- `.replace()`
    
- `str` diziler [ ] ile dilimlenebilecek yinelenebilir nesnelerdir
</details>

In [6]:
key_names = [
    name.replace('olist_', '').replace('_dataset.csv', '').replace('.csv', '') 
    for name in file_names
]
key_names

['product_category_name_translation',
 'products',
 'customers',
 'order_payments',
 'geolocation',
 'order_items',
 'orders',
 'order_reviews',
 'sellers']

### 4. `data` sözlüğünü oluşturun

```python
data = { 
    'sellers': DataFrame1,
    'orders': DataFrame2,
    'order_items': DataFrame3,
    ...
    }
```
Burada `DataFrame1`, `DataFrame2`, ... gerçek `pandas.DataFrame` nesneleri olmalıdır.

<details>
    <summary>▸ İpucu</summary>

`zip()` metodu iki liste üzerinde yinelemek için çok kullanışlıdır
```python
for (x, y) in zip(['a','b','c'], [1,2,3]):
    print(x,y)

# ('a', 1), ('b', 2), ('c', 3) döndürür
    
```
</details>

In [7]:
data = {key: pd.read_csv(path) for key, path in zip(key_names, file_paths)}

# Kontrol etmek için herhangi bir tablonun ilk 5 satırına bakalım
data['orders'].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### 5. `olist/data.py` dosyasındaki `get_data()` metodunu uygulayın

Mantığımızı not defterinden `.py` dosyalarımıza taşımanın zamanı geldi. Bu, bu modül boyunca oluşturacağımız yeni not defterlerinde verileri kolayca yüklememize izin verecek.

Bu modülün klasöründe (bu not defterinin iki seviye yukarısında) `olist/data.py` dosyasını açın ve bu not defterine yazdığınız kodu `get_data()` metoduna taşımaya başlayın. Kodumuzu test etmek için yazdığımız satırları atlayın (`.py` dosyasında listeleri ve DataFrame'leri göstermeye gerek yok).

İşlev çağrıldığında DataFrames içeren sözlüğü döndürmelidir. Bunu csv dosyalarından biriyle deneyip görebilirsiniz:

In [10]:
from olist.data import Olist
Olist().get_data()['sellers'].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Kodunuzu test edin

In [8]:
from nbresult import ChallengeResult
from olist.data import Olist
data = Olist().get_data()
result = ChallengeResult('get_data',
    keys_len=len(data),
    keys=sorted(list(data.keys())),
    columns=sorted(list(data['sellers'].columns)),
    vars_used=Olist.get_data.__code__.co_names
    )
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/data-data-preparation/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 3 items

test_get_data.py::TestGetData::test_columns PASSED                       [ 33%]
test_get_data.py::TestGetData::test_keys PASSED                          [ 66%]
test_get_data.py::TestGetData::test_len PASSED                           [100%]

============================== 3 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/get_data.pickle

git commit -m 'Completed get_data step'

git push origin master



❓Bu kod parçası makinenizde her yerden çalışmalı, sadece bu not defterinden değil.
- Yeni bir terminal açın
- Ana klasörünüze gidin `cd`
- Bir `ipython` oturumu başlatın
- Yukarıdaki iki satır kodu test edin 👆

🏁 Tebrikler!

💾 Şunları commit ve push etmeyi unutmayın: 
* bu `data_preparation.ipynb` not defteri ve challenge klasöründeki `tests/get_data.pickle` test sonuçları
* ayrıca `/olist` klasöründeki `data.py` dosyası